In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import textwrap

@kbench.task(name="counter_factual_weather_learning")
def counter_factual_weather_learning(llm):
    data = {
        'Date': ['2018-01-01', '2018-01-02', '2018-01-03', '2018-01-04', '2018-01-05', '2018-01-06', '2018-01-07'],
        'Temperature': [10, 12, 8, 9, 15, 18, 20],
        'Wind Speed': [3.0, 4.1, 2.5, 5.0, 6.1, 4.5, 7.2],
        'GHI': [200, 250, 150, 399, 350, 450, 500],
        'Pressure': [1012, 1015, 1011, 1008, 1010, 1005, 998]
    }
    df = pd.DataFrame(data)
    data_snippet = df.to_string(index=False)

    prompt = textwrap.dedent(f"""
        You are an agent in a simulation where the laws of meteorology are different. In this world:

        1. If 'Wind Speed' is higher than 5.0, it generates 'Aero-Heat,' adding +10 to the 'Temperature' to get the 'Perceived Temperature'. Otherwise, 'Perceived Temperature' is the same as 'Temperature'.

        2. 'Solar Radiation' (GHI) is actually 'Atmospheric Density.' If it exceeds 400, the 'Pressure' must be read in reverse to get the 'Corrected Pressure' (e.g., 1000 becomes 0001). Otherwise, 'Corrected Pressure' is the same as 'Pressure'.

        Below is a data snippet from the year 2018. Based strictly and only on these two new rules, what is the 'Perceived Temperature' and 'Corrected Pressure' for the last three rows of this data? Ignore your real-world knowledge of weather.

        Data:
        {data_snippet}
    """)

    response = llm.prompt(prompt)

    assessment = kbench.assertions.assess_response_with_judge(
        criteria=[
            "For the row with Date 2018-01-05, the Perceived Temperature is correctly calculated as 25 (15 + 10).",
            "For the row with Date 2018-01-05, the Corrected Pressure is correctly stated as 1010.",
            "For the row with Date 2018-01-06, the Perceived Temperature is correctly stated as 18.",
            "For the row with Date 2018-01-06, the Corrected Pressure is correctly calculated as 5001 (reverse of 1005).",
            "For the row with Date 2018-01-07, the Perceived Temperature is correctly calculated as 30 (20 + 10).",
            "For the row with Date 2018-01-07, the Corrected Pressure is correctly calculated as 899 (reverse of 998).",
        ],
        response_text=response,
        judge_llm=kbench.judge_llm
    )

    if assessment is None:
        kbench.assertions.assert_fail(expectation="Judge LLM failed to return a valid assessment.")
        return

    for result in assessment.results:
        kbench.assertions.assert_true(
            result.passed,
            expectation=f"Criterion '{result.criterion}' failed: {result.reason}"
        )

counter_factual_weather_learning.run(kbench.llm)